# Analisi del dataset Used Cars

[Dati](https://www.kaggle.com/competitions/playground-series-s4e9)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
df_train = pd.read_csv("train.csv", index_col=0)
df_test = pd.read_csv("test.csv", index_col=0)

In [3]:
print(f"Shape del train: {df_train.shape}\n")
print(f"Shape del test: {df_test.shape}")

Shape del train: (188533, 12)

Shape del test: (125690, 11)


In [4]:
df_train.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
id,,,,,,,,,,,,
0,MINI,Cooper S Base,2007,213000,Gasoline,172.0HP 1.6L 4 Cylinder Engine Gasoline Fuel,A/T,Yellow,Gray,None reported,Yes,4200
1,Lincoln,LS V8,2002,143250,Gasoline,252.0HP 3.9L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Beige,At least 1 accident or damage reported,Yes,4999
2,Chevrolet,Silverado 2500 LT,2002,136731,E85 Flex Fuel,320.0HP 5.3L 8 Cylinder Engine Flex Fuel Capab...,A/T,Blue,Gray,None reported,Yes,13900
3,Genesis,G90 5.0 Ultimate,2017,19500,Gasoline,420.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Black,Black,None reported,Yes,45000
4,Mercedes-Benz,Metris Base,2021,7388,Gasoline,208.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,7-Speed A/T,Black,Beige,None reported,Yes,97500


In [5]:
df_test.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title
id,,,,,,,,,,,
188533,Land,Rover LR2 Base,2015,98000,Gasoline,240.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Beige,None reported,Yes
188534,Land,Rover Defender SE,2020,9142,Hybrid,395.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,8-Speed A/T,Silver,Black,None reported,Yes
188535,Ford,Expedition Limited,2022,28121,Gasoline,3.5L V6 24V PDI DOHC Twin Turbo,10-Speed Automatic,White,Ebony,None reported,NaN
188536,Audi,A6 2.0T Sport,2016,61258,Gasoline,2.0 Liter TFSI,Automatic,Silician Yellow,Black,None reported,NaN
188537,Audi,A6 2.0T Premium Plus,2018,59000,Gasoline,252.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,A/T,Gray,Black,None reported,Yes


In [6]:
print("=====================Train=====================\n")
print(df_train.info())
print("\n=====================Test======================\n")
print(df_test.info())

=====================Train=====================

<class 'pandas.core.frame.DataFrame'>
Index: 188533 entries, 0 to 188532
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   brand         188533 non-null  object
 1   model         188533 non-null  object
 2   model_year    188533 non-null  int64 
 3   milage        188533 non-null  int64 
 4   fuel_type     183450 non-null  object
 5   engine        188533 non-null  object
 6   transmission  188533 non-null  object
 7   ext_col       188533 non-null  object
 8   int_col       188533 non-null  object
 9   accident      186081 non-null  object
 10  clean_title   167114 non-null  object
 11  price         188533 non-null  int64 
dtypes: int64(3), object(9)
memory usage: 18.7+ MB
None

=====================Test======================

<class 'pandas.core.frame.DataFrame'>
Index: 125690 entries, 188533 to 314222
Data columns (total 11 columns):
 #   Column        Non-Nul

In [7]:
print(f"Duplicati:\n - Train: {df_train.duplicated().sum()}\n - Test: {df_test.duplicated().sum()}")

Duplicati:
 - Train: 0
 - Test: 0


In [8]:
print(f"Valori nulli:\n - Train:\n{df_train.isnull().sum()}\n\n - Test:\n{df_test.isnull().sum()}")

Valori nulli:
 - Train:
brand               0
model               0
model_year          0
milage              0
fuel_type        5083
engine              0
transmission        0
ext_col             0
int_col             0
accident         2452
clean_title     21419
price               0
dtype: int64

 - Test:
brand               0
model               0
model_year          0
milage              0
fuel_type        3383
engine              0
transmission        0
ext_col             0
int_col             0
accident         1632
clean_title     14239
dtype: int64


In [9]:
from sklearn.impute import SimpleImputer
from sklearn.calibration import LabelEncoder

In [10]:
def process_engine(df):
    if 'engine' in df.columns:
        df['horsepower'] = df['engine'].str.extract(r'(\d+\.\d+)(?=HP)').astype(float)
        df['engine_size'] = df['engine'].str.extract(r'(\d+\.\d+)(?=L)').astype(float)
        df['cylinders'] = df['engine'].str.extract(r'(\d+)\s(?:Cylinder|V\d|Straight)')[0].astype(float)

        df['horsepower'] = round(df['horsepower'].fillna(df['horsepower'].mean()))
        df['engine_size'] = round(df['engine_size'].fillna(df['engine_size'].mean()))
        df['cylinders'] = round(df['cylinders'].fillna(df['cylinders'].mean()))
        
        df = df.drop('engine', axis=1)
    return df

def clean_transmission(value):
    if pd.isna(value): return 'Other'
    val = value.lower()
    if 'cvt' in val: return 'CVT'
    elif 'manual' in val or 'm/t' in val: return 'Manual'
    elif 'auto-shift' in val or 'dct' in val or 'dual shift' in val: return 'Dual-Clutch'
    elif 'a/t' in val or 'automatic' in val or 'overdrive' in val: return 'Automatic'
    elif 'speed' in val and ('mt' in val or 'm/t' in val): return 'Manual'
    else: return 'Other'

def process_transmission(df):
    if 'transmission' in df.columns:
        df['transmission'] = df['transmission'].apply(clean_transmission)
    return df

def preprocess_data(df):
    # Drop colonne inutili
    df = df.drop(columns=['int_col'], errors='ignore')

    # Pulizia fuel_type
    df['fuel_type'] = df['fuel_type'].replace(['–', 'not supported'], np.nan)
    df['fuel_type'] = df['fuel_type'].fillna(df['fuel_type'].mode()[0])

    # Pulizia accident e clean_title
    df['accident'] = df['accident'].fillna('None reported')
    df['clean_title'] = df['clean_title'].fillna('No')

    # Applicazione funzioni modulari
    df = process_engine(df)
    df = process_transmission(df)

    # Label encoding per model e brand
    for col in ['model', 'brand']:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    # One-Hot Encoding su colonne categoriche
    categorical_cols = ['fuel_type', 'transmission', 'ext_col', 'accident', 'clean_title']
    existing_cols = [col for col in categorical_cols if col in df.columns]
    encoder = OneHotEncoder(drop='first', handle_unknown='ignore')
    encoded_data = encoder.fit_transform(df[existing_cols]).toarray()
    cols = encoder.get_feature_names_out(existing_cols)
    print(cols.shape)
    encoded_df = pd.DataFrame(encoded_data, columns=cols, index=df.index)

    non_categorical_cols = df.drop(columns=existing_cols)
    df = pd.concat([non_categorical_cols, encoded_df], axis=1)

    return df


In [11]:
train_df = preprocess_data(df_train)
train_df.head()

(328,)


,brand,model,model_year,milage,price,horsepower,engine_size,cylinders,fuel_type_E85 Flex Fuel,fuel_type_Gasoline,...,ext_col_Wolf Gray,ext_col_Yellow,ext_col_Yulong,ext_col_Yulong White,ext_col_designo Diamond White,ext_col_designo Diamond White Bright,ext_col_designo Diamond White Metallic,ext_col_–,accident_None reported,clean_title_Yes
id,,,,,,,,,,,,,,,,,,,,,
0,31,495,2007,213000,4200,172.0,2.0,4.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,28,930,2002,143250,4999,252.0,4.0,8.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,9,1575,2002,136731,13900,320.0,5.0,8.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,16,758,2017,19500,45000,420.0,5.0,8.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
4,36,1077,2021,7388,97500,208.0,2.0,4.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


In [14]:
X = df_train.drop('price', axis=1)
y = df_train['price']

In [25]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVR

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
# Assicurati che le colonne di df_test_processed siano le stesse di X_processed per una scalatura corretta
# Questa parte è cruciale per evitare errori di shape o colonna mancante
test_df_processed_aligned = df_test[X.columns] # Allinea le colonne di test con quelle di training
df_test_scaled = scaler.transform(test_df_processed_aligned)

# Converti X_scaled e df_test_scaled in DataFrame per mantenere i nomi delle colonne
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)
df_test_scaled_df = pd.DataFrame(df_test_scaled, columns=test_df_processed_aligned.columns, index=df_test.index)


# --- Suddivisione del Dataset (per valutazione locale) ---
X_train, X_val, y_train, y_val = train_test_split(X_scaled_df, y, test_size=0.2, random_state=42)

# --- Addestramento del Modello SVR ---
print("\nInizio addestramento modello SVR iniziale...")
svr_model = SVR(kernel='rbf', C=100, epsilon=0.1)
svr_model.fit(X_train, y_train)

y_pred_val = svr_model.predict(X_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
print(f"RMSE del modello SVR (su set di validazione): {rmse_val:.4f}")

# --- Ottimizzazione degli Iperparametri (Grid Search) ---
print("\nInizio Grid Search per ottimizzazione degli iperparametri...")
param_grid = {
    'C': [1, 10, 100],
    'epsilon': [0.05, 0.1, 0.2],
    'kernel': ['rbf'],
    'gamma': ['scale', 0.01, 0.1]
}

grid_search = GridSearchCV(SVR(), param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

print(f"\nMigliori parametri: {grid_search.best_params_}")
best_svr_model = grid_search.best_estimator_

y_pred_best_val = best_svr_model.predict(X_val)
rmse_best_val = np.sqrt(mean_squared_error(y_val, y_pred_best_val))
print(f"RMSE del modello SVR (migliore da Grid Search su set di validazione): {rmse_best_val:.4f}")

# --- Addestramento Finale e Generazione Submission ---
print("\nAddestramento del modello finale su tutto il set di training processato...")
# Addestra il modello migliore sull'intero set di dati di training SCALATO
best_svr_model.fit(X_scaled_df, y)

# Fai previsioni sul df_test_processed SCALATO per la submission
final_test_predictions = best_svr_model.predict(df_test_scaled_df)

# Assicurati che le previsioni siano non negative
final_test_predictions[final_test_predictions < 0] = 0

# Crea il file di submission
submission_df = pd.DataFrame({'id': test_ids, 'price': final_test_predictions})
submission_df.to_csv('submission.csv', index=False)
print("\nFile 'submission.csv' creato con successo!")


ValueError: could not convert string to float: 'MINI'